# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully referenced by Croissant entity `@id` values.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant` and inspect high-level dataset description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema download URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show dataset description
print(f"{getattr(metadata, 'name', 'No name')}: {getattr(metadata, 'description', 'No description')}")

## 2. Data Overview
This section overviews available record sets and their fields in the dataset.

Note: All entities are referred by their `@id`.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '')}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields: {field_ids}\n")

## 3. Data Extraction
In this section, we extract all data from each record set. All record sets will be loaded as DataFrames. Use the `@id` for each record set and field for future analysis.

In [ ]:
# We'll extract all defined record sets into dataframes, by @id
dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    print(f"RecordSet @id: {rs_id} | Shape: {df.shape}")
    dataframes[rs_id] = df

# Display first rows and columns for the main record set
if record_sets:
    main_rs_id = record_sets[0].id
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, let's perform some EDA tasks: filter numeric fields, normalize numeric values, and group by selected fields. To do so, we must reference all columns and field attributes by their `@id`.

**You can replace these field IDs by other field IDs as needed depending on which fields are available.**

In [ ]:
# Select record set and one of its numeric fields by @id
from pprint import pprint

# For demonstration, assume the main record set contains a numeric field with @id 'cr:peptide_count' and a group/categorical field 'cr:accession'.
# Adjust the field IDs below as needed based on the real field names in the data overview output.
record_set_id = main_rs_id
df = dataframes[record_set_id]

# Try to find a numeric column (e.g., 'peptide_count', 'coverage', 'MW')
print("Available columns for EDA:", df.columns.tolist())

# Let's pick a likely numeric field by @id (change as needed from your schema):
numeric_field_id = None
for col in df.columns:
    if 'coverage' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Default to first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    numeric_field_id = df.columns[0]  # fallback
print(f"Selected numeric field for analysis (by @id): {numeric_field_id}\n")

# Filter using an example threshold (e.g., top proteins with coverage > 10)
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try grouping by an accession/description field
    possible_group_fields = [col for col in df.columns if 'accession' in col.lower() or 'description' in col.lower() or 'sample' in col.lower()]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
        pprint(grouped_df.head())
else:
    print(f"Field {numeric_field_id} is not numeric, please choose another.")

## 5. Visualization
Visualizing the distribution of selected numeric field and top groups (by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Bar plot of top 10 groups by mean value (if grouping field exists)
if 'grouped_df' in locals() and group_field is not None:
    top_groups = grouped_df.sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_groups.values, y=top_groups.index)
    plt.title(f"Top 10 {group_field} by average {numeric_field_id}")
    plt.xlabel(f"Average {numeric_field_id}")
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² dataset using `mlcroissant` and referenced all structure elements by their `@id`. We:
- Loaded dataset metadata and reviewed available record sets and fields (@id).
- Extracted record sets to DataFrames and identified numeric/categorical fields.
- Applied basic filtering, normalization, and grouped aggregation for EDA.
- Visualized field distributions and top groupings.

**Next Steps:** Use this notebook as a foundation for deeper analysis by substituting your fields of interest, refining groupings, and developing advanced visualizations or predictive models.